# AYA Edge 0.2 — FineWeb-Edu Experimental Training

**Status:** Experimental research notebook.  
**Model:** ~310 M parameter AYA Edge 0.2 candidate (MLA-inspired + top-k MoE).  
**Data:** HuggingFaceFW/fineweb-edu (ODC-By).  
**Tokenizer:** Locked 32 768-vocab byte-level BPE (specials 0–15 / byte fallback 16–271 / BPE 272–32767).

> **License & responsibility**  
> FineWeb-Edu is released under ODC-By. You are responsible for complying with the terms before any training or redistribution of derived models.  
> This notebook is **not** an approved production training run. The repository acquisition pipeline still marks FineWeb as license-blocked for formal Stage-1 jobs. Use this only for research / architecture validation.

Recommended hardware: 1× or 2× Tesla T4 (16 GB) or better. The notebook streams a controllable subset so it can finish inside a typical Kaggle / Colab session.

## 0. Configuration (edit these cells)

In [ ]:
# ============== USER-TUNABLE KNOBS ==============
MAX_DOCS          = 20_000      # number of FineWeb-Edu documents to stream (raise carefully)
SEQ_LEN           = 256         # tokens per packed sequence (512 is possible on T4 with batch=1)
BATCH_SIZE        = 1
GRAD_ACCUM        = 16          # effective batch = BATCH_SIZE * GRAD_ACCUM
MAX_STEPS         = 1_000       # training steps (increase for longer runs)
LEARNING_RATE     = 2e-4
WARMUP_STEPS      = 100
CHECKPOINT_EVERY  = 200
VAL_EVERY         = 100
SEED              = 42

# FineWeb-Edu subset. "sample-10BT" is convenient; any CC-MAIN-* dump also works.
FINEWEB_NAME      = "sample-10BT"
FINEWEB_SPLIT     = "train"

# Where to write artefacts
OUTPUT_DIR_NAME   = "aya_edge_fineweb_edu_run"
# =================================================

## 1. Environment & repository

In [ ]:
import json, os, platform, shutil, subprocess, sys, time, math, hashlib
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
REPO = WORK / 'Aya-Edge-0.1'
REPO_URL = 'https://github.com/hygienhub03-dev/Aya-Edge-0.1.git'

if not (REPO / 'pyproject.toml').exists():
    print('Cloning repository...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)
else:
    print('Repository already present')

os.chdir(REPO)
print('Working directory:', Path.cwd())
print('HEAD:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

# Core + data extras
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'datasets', 'huggingface_hub', 'tokenizers', 'pyarrow', 'tqdm'], check=True)
print('Dependencies ready')

In [ ]:
import torch
from torch.amp import GradScaler, autocast

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Python :', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('Device :', device)
if device.type == 'cuda':
    print('GPU    :', torch.cuda.get_device_name(0))
    print('VRAM   :', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), 'GiB')
    print('BF16   :', torch.cuda.is_bf16_supported())
print('Disk free GiB:', round(shutil.disk_usage(WORK).free / 2**30, 2))

## 2. Tokenizer (locked 32 768 layout)

We first train a clearly-labelled **NON-PRODUCTION** tokenizer on a small FineWeb-Edu sample so the dual-T4 / single-T4 path has a real 32k artifact.  
Later you can replace it with a fully approved production tokenizer.

In [ ]:
from datasets import load_dataset
from aya_edge.tokenizer.bpe import (
    train_production_bpe, write_manifest, sha256_file,
    PRODUCTION_VOCAB_SIZE, SPECIAL_TOKENS
)
from aya_edge.tokenizer import ProductionBPETokenizer

ART = Path('artifacts')
ART.mkdir(exist_ok=True)
TOK_PATH = ART / 'tokenizer_fineweb_nonprod.json'
MAN_PATH = ART / 'tokenizer_fineweb_nonprod.manifest.json'

# Stream a few thousand docs purely for tokenizer training
print('Streaming small FineWeb-Edu sample for tokenizer training...')
ds_tok = load_dataset(
    'HuggingFaceFW/fineweb-edu',
    name=FINEWEB_NAME,
    split=FINEWEB_SPLIT,
    streaming=True,
)
tok_texts = []
for i, row in enumerate(ds_tok):
    text = (row.get('text') or '').strip()
    if len(text) > 64:
        tok_texts.append(text)
    if len(tok_texts) >= 4_000:          # enough to produce a stable 32k vocab
        break

tok_file = ART / 'fineweb_tok_sample.txt'
tok_file.write_text('\n'.join(tok_texts), encoding='utf-8')
print(f'Wrote {len(tok_texts)} documents ({tok_file.stat().st_size/1e6:.1f} MB) for tokenizer training')

print('Training 32 768-vocab byte-level BPE (this takes 1–3 min)...')
train_production_bpe([tok_file], TOK_PATH, min_frequency=2)

# Minimal non-prod manifest
impl = 'fineweb-edu-nonprod-' + sha256_file(TOK_PATH)[:12]
manifest = write_manifest(
    TOK_PATH, MAN_PATH,
    corpus_manifest_sha256=hashlib.sha256(b'fineweb-edu-sample-for-tok').hexdigest(),
    implementation_commit=impl,
)
data = manifest.to_dict()
data['label'] = 'NON-PRODUCTION'
data['source'] = f'HuggingFaceFW/fineweb-edu/{FINEWEB_NAME}'
MAN_PATH.write_text(json.dumps(data, indent=2) + '\n')

tokenizer = ProductionBPETokenizer.load(TOK_PATH)
print('Tokenizer ready')
print('  vocab_size :', tokenizer.vocab_size)
print('  specials   :', list(SPECIAL_TOKENS)[:4], '...')
print('  sha256     :', data['tokenizer_sha256'][:16], '...')
print('  layout     : 0-15 specials / 16-271 byte fallback / 272-32767 BPE')

## 3. Stream & pack FineWeb-Edu for training

In [ ]:
from tqdm.auto import tqdm

print(f'Streaming up to {MAX_DOCS} FineWeb-Edu documents...')
ds = load_dataset(
    'HuggingFaceFW/fineweb-edu',
    name=FINEWEB_NAME,
    split=FINEWEB_SPLIT,
    streaming=True,
)

all_ids = []
docs_used = 0
for row in tqdm(ds, total=MAX_DOCS, desc='tokenizing'):
    text = (row.get('text') or '').strip()
    if len(text) < 32:
        continue
    ids = tokenizer.encode(text, add_eos=True)
    all_ids.extend(ids)
    docs_used += 1
    if docs_used >= MAX_DOCS:
        break

print(f'Documents used : {docs_used:,}')
print(f'Total tokens   : {len(all_ids):,}')

# Pack into fixed-length sequences (drop incomplete tail)
usable = (len(all_ids) // SEQ_LEN) * SEQ_LEN
if usable < SEQ_LEN * 8:
    raise RuntimeError('Not enough tokens after packing; increase MAX_DOCS')

tokens = torch.tensor(all_ids[:usable], dtype=torch.long).view(-1, SEQ_LEN)
n_seq = tokens.shape[0]
# Simple 95 / 5 split
n_val = max(8, n_seq // 20)
train_tokens = tokens[:-n_val]
val_tokens   = tokens[-n_val:]
print(f'Packed sequences: train={train_tokens.shape[0]:,}  val={val_tokens.shape[0]:,}  seq_len={SEQ_LEN}')

## 4. Model construction

In [ ]:
import yaml
from aya_edge import AyaEdgeConfig, AyaEdgeForCausalLM, count_parameters, count_active_parameters
from aya_edge.training import set_seed, save_checkpoint

raw = yaml.safe_load(Path('configs/aya_edge_0_2.yaml').read_text())
cfg = AyaEdgeConfig(**raw['model'])
# Make sure model matches the tokenizer we just trained
assert cfg.vocab_size == PRODUCTION_VOCAB_SIZE
tokenizer.assert_model_alignment(cfg)

set_seed(SEED)
model = AyaEdgeForCausalLM(cfg).to(device)
print('Configuration :', cfg.to_dict())
print(f'Total parameters          : {count_parameters(model):,}')
print(f'Structural active estimate: {count_active_parameters(model):,}')
print('Prototype status: research candidate — not production-ready')

## 5. Optimizer, scheduler, AMP

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.1,
    betas=(0.9, 0.95),
)

def lr_lambda(step):
    if step < WARMUP_STEPS:
        return max(step, 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * min(1.0, max(0.0, progress))))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
use_amp = device.type == 'cuda'
scaler = GradScaler('cuda', enabled=use_amp)
print({'device': str(device), 'amp': use_amp, 'effective_batch': BATCH_SIZE * GRAD_ACCUM})

## 6. Training loop

In [ ]:
OUT = WORK / OUTPUT_DIR_NAME
OUT.mkdir(parents=True, exist_ok=True)
metrics_log = []

def evaluate(n_batches=8):
    model.eval()
    losses = []
    with torch.no_grad():
        for i in range(min(n_batches, len(val_tokens))):
            batch = val_tokens[i:i+1].to(device)
            with autocast('cuda', dtype=torch.float16, enabled=use_amp):
                out = model(batch, labels=batch)
            losses.append(float(out['loss']))
    model.train()
    return sum(losses) / max(1, len(losses))

model.train()
start = time.time()
step = 0
print(f'Starting training for {MAX_STEPS} steps ...')

while step < MAX_STEPS:
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    for micro in range(GRAD_ACCUM):
        idx = (step * GRAD_ACCUM + micro) % len(train_tokens)
        batch = train_tokens[idx:idx+BATCH_SIZE].to(device)
        with autocast('cuda', dtype=torch.float16, enabled=use_amp):
            out = model(batch, labels=batch)
            loss = (out['loss'] + 0.01 * out['aux_loss']) / GRAD_ACCUM
        scaler.scale(loss).backward()
        total_loss += float(loss.detach())

    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    if not torch.isfinite(grad_norm):
        raise FloatingPointError(f'non-finite grad at step {step}')
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()
    step += 1

    if step == 1 or step % 20 == 0 or step == MAX_STEPS:
        elapsed = max(time.time() - start, 1e-9)
        tok_per_s = step * BATCH_SIZE * GRAD_ACCUM * SEQ_LEN / elapsed
        entry = {
            'step': step,
            'loss': round(total_loss, 4),
            'grad_norm': round(float(grad_norm), 3),
            'lr': scheduler.get_last_lr()[0],
            'tok/s': round(tok_per_s, 1),
            'peak_mem_GiB': round(torch.cuda.max_memory_allocated()/2**30, 2) if device.type=='cuda' else None,
        }
        if step == 1 or step % VAL_EVERY == 0 or step == MAX_STEPS:
            entry['val_loss'] = round(evaluate(), 4)
        metrics_log.append(entry)
        print(entry)

    if step % CHECKPOINT_EVERY == 0 or step == MAX_STEPS:
        ckpt = OUT / f'aya_edge_fineweb_step{step:05d}.pt'
        save_checkpoint(
            ckpt, model, optimizer, scheduler=scheduler, scaler=scaler,
            step=step,
            tokens_processed=step * BATCH_SIZE * GRAD_ACCUM * SEQ_LEN,
            config=cfg,
            dataset_version=f'HuggingFaceFW/fineweb-edu/{FINEWEB_NAME} (streamed {docs_used} docs)',
        )
        print('Checkpoint written:', ckpt)

print('Training finished.')

## 7. Quick generation sample

In [ ]:
model.eval()
prompt = "The scientific method is a systematic way of"
ids = tokenizer.encode(prompt)
input_ids = torch.tensor([ids], dtype=torch.long, device=device)
with torch.no_grad():
    generated = model.generate(
        input_ids,
        max_new_tokens=64,
        temperature=0.8,
        top_k=40,
        eos_token_id=tokenizer._backend.token_to_id('<|eos|>'),
    )
print('Prompt :', prompt)
print('Generated:', tokenizer.decode(generated[0].tolist()))

## 8. Final report

In [ ]:
report = {
    'status': 'experimental FineWeb-Edu run finished',
    'git_head': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
    'model_parameters': count_parameters(model),
    'active_estimate': count_active_parameters(model),
    'tokenizer_sha256': data['tokenizer_sha256'],
    'tokenizer_label': 'NON-PRODUCTION',
    'id_layout': '0-15 specials / 16-271 byte fallback / 272-32767 BPE',
    'dataset': f'HuggingFaceFW/fineweb-edu/{FINEWEB_NAME}',
    'docs_streamed': docs_used,
    'tokens_packed': int(usable),
    'seq_len': SEQ_LEN,
    'max_steps': MAX_STEPS,
    'final_metrics': metrics_log[-1] if metrics_log else None,
    'output_dir': str(OUT),
    'warnings': [
        'This is NOT an approved production training run.',
        'Tokenizer is labelled NON-PRODUCTION and was trained on a tiny sample.',
        'FineWeb-Edu license (ODC-By) compliance is the responsibility of the user.',
        'Model remains a research prototype (Python MoE dispatch, no KV cache).',
    ],
}
print(json.dumps(report, indent=2))
(OUT / 'run_report.json').write_text(json.dumps(report, indent=2) + '\n')
print('Report saved to', OUT / 'run_report.json')